# CV Masterclass 17: Multi-Object Tracking (MOT) & Kinematics

Welcome to Notebook 17. In previous modules, we mastered the art of "seeing"—detecting objects in a static frame. Today, we master the art of **Temporal Association**—following those objects through time and space.

Multi-Object Tracking (MOT) is the backbone of autonomous driving, surveillance, and sports analytics. It asks the fundamental question: *"Is the person at $(x_1, y_1)$ in Frame 1 the same person at $(x_2, y_2)$ in Frame 2?"*

---

## 🎓 Socratic Deep Check
Ponder these questions before we begin:

> *"In a crowded subway station, two people walk past each other, and their bounding boxes overlap (occlusion). If they have similar clothes, how does a computer know they didn't swap identities when they separated?"*

> *"If a camera's frame rate drops from 30 FPS to 5 FPS, why does tracking become exponentially harder? What physical assumption are we breaking?"*

---

## Table of Contents
1. **The Tracking-by-Detection Paradigm:** Separating Detection from Association.
2. **The Kalman Filter:** The Mathematics of Kinematic Prediction.
3. **Bipartite Matching:** The Hungarian Algorithm and the Cost Matrix.
4. **DeepSORT:** Integrating Re-Identification (ReID) Embeddings.
5. **ByteTrack:** The Power of Low-Confidence Detections.
6. **Implementation:** Building a Kalman Tracker in Numpy.
7. **Common Pitfalls:** ID Switches and Fragmentation.

## 1. The Tracking-by-Detection Paradigm

Modern MOT systems almost exclusively use the **Tracking-by-Detection** paradigm. Instead of trying to track raw pixels (like Optical Flow), we decompose the problem:

1.  **Detection Step:** A high-performance detector (YOLOv8, RT-DETR) identifies all objects in frame $T$.
2.  **Association Step:** A tracker associates the detections in frame $T$ with existing "tracklets" from frame $T-1$.

### Why decouple them?
- **Robustness:** If a detector misses a frame (false negative), a motion model can still predict where the object *should* be.
- **Scalability:** Detectors are expensive; association (math) is cheap. We can run detection at 10 FPS and tracking at 60 FPS to maintain smooth trajectories.

## 2. The Kalman Filter: Kinematic Math

The **Kalman Filter** is an optimal estimation algorithm that predicts the state of a system (e.g., a bounding box) by combining a noisy measurement with a physics-based prediction.

### The State Vector
For a bounding box, we represent the state $x$ as:
$$ x = [u, v, a, h, \dot{u}, \dot{v}, \dot{a}, \dot{h}]^T $$
Where:
- $(u, v)$: Center coordinates.
- $a$: Aspect ratio ($w/h$).
- $h$: Height.
- $(\dot{u}, \dot{v}, \dot{a}, \dot{h})$: The respective velocities (first derivatives).

### 1. Prediction (The Physics)
We assume constant velocity. The next state $x'$ is predicted using the **State Transition Matrix** $F$:
$$ x_{k|k-1} = F x_{k-1|k-1} $$
$$ P_{k|k-1} = F P_{k-1|k-1} F^T + Q $$
Where $P$ is the covariance (uncertainty) and $Q$ is process noise.

### 2. Update (The Measurement)
When a new YOLO detection arrive $(z)$, we update our prediction:
1.  **Innovation:** $y = z - Hx'$ (Difference between measurement and prediction).
2.  **Kalman Gain:** $K$ (How much should we trust the measurement vs. the prediction?).
3.  **Corrected State:** $x = x' + Ky$.

In [ ]:
import numpy as np
# Install filterpy if not available
try:
    from filterpy.kalman import KalmanFilter
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'filterpy', '--quiet'])
    from filterpy.kalman import KalmanFilter

print("filterpy available ✓")

class KalmanBoxTracker:
    """
    Represents the internal state of individual tracked objects observed as bbox.
    """
    def __init__(self, bbox):
        # Define constant velocity model
        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        self.kf.F = np.array([[1,0,0,0,1,0,0],[0,1,0,0,0,1,0],[0,0,1,0,0,0,1],[0,0,0,1,0,0,0],  \
                             [0,0,0,0,1,0,0],[0,0,0,0,0,1,0],[0,0,0,0,0,0,1]])
        self.kf.H = np.array([[1,0,0,0,0,0,0],[0,1,0,0,0,0,0],[0,0,1,0,0,0,0],[0,0,0,1,0,0,0]])

        self.kf.R[2:,2:] *= 10. # Measurement noise for scale/aspect ratio
        self.kf.P[4:,4:] *= 1000. # High uncertainty to unobservable initial velocities
        self.kf.P *= 10.
        self.kf.Q[-1,-1] *= 0.01
        self.kf.Q[4:,4:] *= 0.01

        self.kf.x[:4] = self.convert_bbox_to_z(bbox)
        self.time_since_update = 0
        self.id = 0 # To be assigned by tracker
        self.history = []

    def update(self, bbox):
        """Updates the state vector with observed bbox."""
        self.time_since_update = 0
        self.history = []
        self.kf.update(self.convert_bbox_to_z(bbox))

    def predict(self):
        """Advances the state vector and returns the predicted bounding box estimate."""
        if((self.kf.x[6]+self.kf.x[2])<=0):
            self.kf.x[6] *= 0.0
        self.kf.predict()
        self.time_since_update += 1
        return self.convert_x_to_bbox(self.kf.x)

    @staticmethod
    def convert_bbox_to_z(bbox):
        """Takes [x1,y1,x2,y2] and returns [u,v,s,r] (center x, y, scale, aspect ratio)"""
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        x = bbox[0] + w/2.
        y = bbox[1] + h/2.
        s = w * h # scale is area
        r = w / float(h)
        return np.array([x, y, s, r]).reshape((4, 1))

    @staticmethod
    def convert_x_to_bbox(x, score=None):
        """Takes [u,v,s,r] and returns [x1,y1,x2,y2]"""
        w = np.sqrt(x[2] * x[3])
        h = x[2] / w
        if(score==None):
            return np.array([x[0]-w/2.,x[1]-h/2.,x[0]+w/2.,x[1]+h/2.]).reshape((1,4))
        else:
            return np.array([x[0]-w/2.,x[1]-h/2.,x[0]+w/2.,x[1]+h/2.,score]).reshape((1,5))

# TEST IT
initial_bbox = [100, 100, 200, 200] # x1, y1, x2, y2
tracker = KalmanBoxTracker(initial_bbox)

# 1. Predict next state
prediction = tracker.predict()
print(f"Initial Prediction (should be nearly same): {prediction}")

# 2. Simulate movement and update
new_bbox = [110, 110, 210, 210] # Moved +10 pixels
tracker.update(new_bbox)

# 3. Predict again - Kalman should realize there is a velocity
prediction_2 = tracker.predict()
print(f"Prediction 2 (after movement): {prediction_2}")

assert prediction_2[0,0] > 100 # X-coord should have increased
print("Kalman Kinematics Verified!")

## 3. Bipartite Matching: The Hungarian Algorithm

Once we have $N$ predicted tracks and $M$ new detections, how do we assign them? This is a **Bipartite Matching** problem.

### The Cost Matrix
We build a matrix $C$ where $C_{ij}$ represents the "distance" between track $i$ and detection $j$. For Simple Online Realtime Tracking (SORT), we use **IoU Distance**:
$$ C_{ij} = 1 - IoU(Track_i, Detection_j) $$

### The Hungarian Algorithm (Kuhn-Munkres)
The Hungarian algorithm solves the assignment problem in $O(N^3)$ time by finding the combination of pairs that **minimizes total global cost**. 

| | Det 1 | Det 2 | Det 3 |
|---|---|---|---|
| **Track A** | 0.1 | 0.8 | 0.9 |
| **Track B** | 0.7 | 0.2 | 0.9 |
| **Track C** | 0.9 | 0.9 | 0.3 |

*Optimal Assignment: (A,1), (B,2), (C,3).*

In [ ]:
from scipy.optimize import linear_sum_assignment

def associate_detections_to_trackers(detections, trackers, iou_threshold=0.3):
    """
    Assigns detections to tracked object (both represented as bounding boxes)
    Returns 3 lists: matches, unmatched_detections, unmatched_trackers
    """
    if(len(trackers)==0):
        return np.empty((0,2),dtype=int), np.arange(len(detections)), np.empty((0,5),dtype=int)

    iou_matrix = np.zeros((len(detections),len(trackers)),dtype=np.float32)

    for d,det in enumerate(detections):
        for t,trk in enumerate(trackers):
            # Placeholder for IoU calculation (usually separate function)
            # We simulate a cost matrix here for the test
            pass 

    # Using Scipy's Hungarian Implementation
    matched_indices = linear_sum_assignment(iou_matrix)
    
    # Note: In production, we filter matches by iou_threshold
    return matched_indices

# TEST IT
cost_matrix = np.array([
    [0.1, 0.9],
    [0.8, 0.2]
])
row_ind, col_ind = linear_sum_assignment(cost_matrix)
print(f"Row Assignments: {row_ind}")
print(f"Col Assignments: {col_ind}")

assert col_ind[0] == 0 and col_ind[1] == 1
print("Hungarian Matching Logic Verified!")

## 4. DeepSORT: Adding Appearance

The original SORT algorithm fails during **long-term occlusion**. If a person walks behind a pillar for 2 seconds, the Kalman filter's uncertainty $(P)$ grows so large that it might "latch onto" any moving object nearby when the person reappears.

### Deep Appearance Descriptor
DeepSORT solves this by adding a **Re-Identification (ReID)** head. For every detection, we extract a feature vector (embedding) $e$. The cost matrix now becomes a weighted sum:
$$ Cost = \lambda (Cost_{IoU}) + (1-\lambda) (Cost_{Cosine}) $$

Even if the bounding boxes don't overlap (IoU=0), if the person's blue shirt matches the embedding of a lost track, the identity is preserved!

## 5. ByteTrack: Every Box Matters

ByteTrack (2021) changed the game by challenging a standard assumption: *"Throw away detections with low confidence scores (< 0.5)".*

**The Problem:** Low scores often happen when an object is partially occluded. If you throw them away, the track breaks (Fragmentation).

**The Byte Solution:**
1.  **First Match:** Match high-score detections to tracklets using IoU + Motion.
2.  **Second Match:** Take the *unmatched* tracklets and try to match them against the **low-score** detections.
3.  If they match, it means the object is just obscured, not gone!
4.  This simple logic dramatically reduced ID switches and reached SOTA on MOTChallenge benchmarks.

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

def bytetrack_two_pass_matching(tracks, high_dets, low_dets,
                                 high_thresh=0.5, low_iou_thresh=0.3):
    """
    ByteTrack's core two-pass association.
    tracks: list of active track dicts with 'pred_box' and 'id'
    high_dets: list of high-confidence detections (score > high_thresh)
    low_dets: list of low-confidence detections
    Returns: (matched_track_det_pairs, unmatched_tracks)
    """
    
    def iou(box1, box2):
        ix1, iy1 = max(box1[0], box2[0]), max(box1[1], box2[1])
        ix2, iy2 = min(box1[2], box2[2]), min(box1[3], box2[3])
        inter = max(0, ix2-ix1) * max(0, iy2-iy1)
        a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
        a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
        return inter / (a1 + a2 - inter + 1e-6)
    
    matched_pairs = []
    remaining_tracks = list(range(len(tracks)))
    
    # --- PASS 1: Match active tracks to HIGH-score detections ---
    if tracks and high_dets:
        cost_1 = np.array([[1.0 - iou(t['pred_box'], d['box'])
                            for d in high_dets] for t in tracks])
        row_ind, col_ind = linear_sum_assignment(cost_1)
        
        assigned_tracks = set()
        for r, c in zip(row_ind, col_ind):
            if cost_1[r, c] < (1.0 - low_iou_thresh):
                matched_pairs.append((tracks[r]['id'], high_dets[c]))
                assigned_tracks.add(r)
        
        remaining_tracks = [i for i in range(len(tracks)) if i not in assigned_tracks]
    
    # --- PASS 2: Match UNMATCHED tracks to LOW-score detections ---
    # Key ByteTrack insight: low-score dets often mean the object
    # is partially occluded — exactly when we need to maintain the track!
    re_matched_in_pass2 = set()
    if remaining_tracks and low_dets:
        cost_2 = np.array([[1.0 - iou(tracks[i]['pred_box'], d['box'])
                            for d in low_dets] for i in remaining_tracks])
        row_ind2, col_ind2 = linear_sum_assignment(cost_2)
        
        for r, c in zip(row_ind2, col_ind2):
            if cost_2[r, c] < (1.0 - low_iou_thresh):
                track_idx = remaining_tracks[r]
                matched_pairs.append((tracks[track_idx]['id'], low_dets[c]))
                re_matched_in_pass2.add(r)
        
        remaining_tracks = [remaining_tracks[i] for i in range(len(remaining_tracks))
                            if i not in re_matched_in_pass2]
    
    unmatched_tracks = [tracks[i] for i in remaining_tracks]
    return matched_pairs, unmatched_tracks

# TEST IT
tracks = [
    {'id': 1, 'pred_box': [10, 10, 50, 50]},   # Person 1
    {'id': 2, 'pred_box': [100, 100, 150, 150]} # Person 2
]
high_dets = [{'box': [12, 12, 52, 52], 'score': 0.9}]  # Close to Track 1
low_dets = [{'box': [98, 98, 148, 148], 'score': 0.3}]  # Close to Track 2 (occluded!)

matched, unmatched = bytetrack_two_pass_matching(tracks, high_dets, low_dets)

print(f"Matched pairs: {[(m[0], 'Detection at '+str(m[1]['box'])) for m in matched]}")
print(f"Unmatched tracks: {[t['id'] for t in unmatched]}")

# Track 1 matched high-conf det in Pass 1, Track 2 matched low-conf det in Pass 2
assert len(matched) == 2, "ByteTrack must recover both tracks!"
assert len(unmatched) == 0, "No tracks should be lost!"
print("ByteTrack two-pass verified: low-confidence occluded detection rescued Track 2 ✓")

### COMMON PITFALLS
*   **ID Switching:** Caused by poor association or crowding. Usually mitigated by adding Appearance (ReID).
*   **Track Fragmentation:** When an object is occluded and the tracker thinks it's a new object when it reappears. ByteTrack is the cure for this.
*   **The Velocity Lag:** Kalman filters assume constant velocity. In sports (like basketball) where players change direction instantly, the Kalman prediction will "overshoot", leading to lost tracks. Increase process noise $(Q)$ to compensate.

### 🎓 DEEP QUESTION ANSWERED

**Q:** *If a camera's frame rate drops, why does tracking become harder?*

**A:** Because we are violating the **Linearity Assumption**. The Kalman filter assumes movement is linear $x = v \cdot t$. As $\Delta t$ increases, the probability that the object turned, stopped, or accelerated increases. This makes the search area for association grow exponentially, leading to "Identity Swapping" with nearby objects.

**Q:** *Can MOT work without a detector?*

**A:** Yes, using **Generative Tracking** or **Optical Flow**. However, these suffer from "Drift"—the tracker slowly accumulates errors until it's tracking a patch of background. Modern "Tracking-by-Detection" is the standard because it "re-anchors" the tracker to reality every single frame.